## [节点配置](https://docs.langchain.com/oss/javascript/langgraph/graph-api-nodes)

### [异常处理](https://docs.langchain.com/oss/python/langgraph/fault-tolerance#error-handling)

In [9]:
from langgraph.errors import NodeError
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command, RetryPolicy
from typing_extensions import TypedDict


class State(TypedDict):
    query: str
    status: str

def query_service(state: State):
    print("开始查询...")
    raise ConnectionError("连接超时")

def query_error_handler(state: State, error: NodeError) -> Command:
    return Command(
        update={"status": f"查询失败: {error.error}"},
        goto="finish",
    )

def finish(state: State):
    return state


builder = StateGraph(State)
builder.add_node(
    query_service,
    retry_policy=RetryPolicy(max_attempts=3, retry_on=ConnectionError),
    error_handler=query_error_handler,
)
builder.add_node(finish)
builder.add_edge(START, "query_service")
builder.add_edge("finish", END)
graph = builder.compile()

result = graph.invoke({})
print(result)

开始查询...
开始查询...
开始查询...
{'status': '查询失败: 连接超时'}
